In [1]:
import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from collections import Counter
import glob

# Set up the dataset path
dataset_path = r'e:\skin_care\data\raw\skin_problems'

print("🔍 Exploring Skin Problems Dataset")
print("=" * 50)

# 1. Load and examine the data.yaml file
yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path, 'r') as file:
    data_config = yaml.safe_load(file)

print("📋 Dataset Configuration:")
print(f"Number of classes: {data_config.get('nc', 'Not specified')}")
print(f"Class names: {data_config.get('names', 'Not specified')}")
print(f"Train path: {data_config.get('train', 'Not specified')}")
print(f"Validation path: {data_config.get('val', 'Not specified')}")
print(f"Test path: {data_config.get('test', 'Not specified')}")
print()

# 2. Analyze dataset structure
splits = ['train', 'valid', 'test']
dataset_stats = {}

for split in splits:
    split_path = os.path.join(dataset_path, split)
    if os.path.exists(split_path):
        images_path = os.path.join(split_path, 'images')
        labels_path = os.path.join(split_path, 'labels')
        
        # Count images and labels
        images = glob.glob(os.path.join(images_path, '*'))
        labels = glob.glob(os.path.join(labels_path, '*.txt'))
        
        dataset_stats[split] = {
            'images': len(images),
            'labels': len(labels),
            'images_path': images_path,
            'labels_path': labels_path
        }

print("📊 Dataset Split Statistics:")
stats_df = pd.DataFrame(dataset_stats).T
print(stats_df)
print()

# 3. Analyze image properties
def analyze_images(split='train', sample_size=50):
    """Analyze image properties for a given split"""
    if split not in dataset_stats:
        return
    
    images_path = dataset_stats[split]['images_path']
    image_files = glob.glob(os.path.join(images_path, '*'))[:sample_size]
    
    widths, heights, channels = [], [], []
    file_sizes = []
    
    print(f"🖼️  Analyzing {len(image_files)} images from {split} set...")
    
    for img_path in image_files:
        try:
            # Get file size
            file_size = os.path.getsize(img_path) / 1024  # KB
            file_sizes.append(file_size)
            
            # Get image dimensions
            with Image.open(img_path) as img:
                width, height = img.size
                widths.append(width)
                heights.append(height)
                channels.append(len(img.getbands()))
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
    
    return {
        'widths': widths,
        'heights': heights,
        'channels': channels,
        'file_sizes': file_sizes
    }

# Analyze train set images
train_analysis = analyze_images('train', sample_size=100)

if train_analysis:
    print("📏 Image Dimensions Analysis (Train Set Sample):")
    print(f"Width - Min: {min(train_analysis['widths'])}, Max: {max(train_analysis['widths'])}, Mean: {np.mean(train_analysis['widths']):.1f}")
    print(f"Height - Min: {min(train_analysis['heights'])}, Max: {max(train_analysis['heights'])}, Mean: {np.mean(train_analysis['heights']):.1f}")
    print(f"Channels - Unique values: {set(train_analysis['channels'])}")
    print(f"File sizes - Min: {min(train_analysis['file_sizes']):.1f}KB, Max: {max(train_analysis['file_sizes']):.1f}KB, Mean: {np.mean(train_analysis['file_sizes']):.1f}KB")
    print()

🔍 Exploring Skin Problems Dataset
📋 Dataset Configuration:
Number of classes: 5
Class names: ['Acne', 'Dark Circle', 'Pigmentation', 'Spots', 'Wrinkle']
Train path: ../train/images
Validation path: ../valid/images
Test path: ../test/images

📊 Dataset Split Statistics:
      images labels                                       images_path  \
train    782    782  e:\skin_care\data\raw\skin_problems\train\images   
valid     50     50  e:\skin_care\data\raw\skin_problems\valid\images   
test      30     30   e:\skin_care\data\raw\skin_problems\test\images   

                                            labels_path  
train  e:\skin_care\data\raw\skin_problems\train\labels  
valid  e:\skin_care\data\raw\skin_problems\valid\labels  
test    e:\skin_care\data\raw\skin_problems\test\labels  

🖼️  Analyzing 100 images from train set...
📏 Image Dimensions Analysis (Train Set Sample):
Width - Min: 640, Max: 640, Mean: 640.0
Height - Min: 640, Max: 640, Mean: 640.0
Channels - Unique values: {3}
Fil